# MobileNet Batch Inference with TaskVine

This tutorial runs CPU-only MobileNetV2 inference over a manifest-defined image dataset. The default `wikimedia_24` profile is a small, self-contained smoke test; a future S3 profile can stage a larger image directory and manifest without changing the workflow.

The same classification workload supports three execution modes:

1. in the notebook process without TaskVine workers;
2. distributed as ordinary TaskVine `PythonTask` tasks; or
3. distributed through TaskVine's stateful serverless library model.


## Step 1 — Choose an execution mode

Set `EXECUTION_MODE` below to `in-process`, `python-task`, or `stateful-serverless`. An environment variable named `MOBILENET_EXECUTION_MODE` takes precedence when it is supplied through Floability's `--env-vars` option. Otherwise, the value in this first configuration cell is used.

Launch `in-process` mode with `--no-worker`. The distributed modes must be launched without that option.


In [ ]:
# EXECUTION_MODE = "in-process"
# EXECUTION_MODE = "python-task"
EXECUTION_MODE = "stateful-serverless"

BATCH_SIZE = 4
TOP_K = 5
LIBRARY_NAME = "mobilenetv2-inference"


## Step 2 — Import the shared helpers and locate staged inputs

Floability runs the notebook inside the staged workflow directory and communicates with it through environment variables. Those Floability and TaskVine setup steps remain visible below. Reusable data validation, inference, and output utilities live in `mobilenet_helpers.py`. Registering the helper module by value allows its imported worker functions to be serialized without installing this workflow-specific module in the packed environment.


In [ ]:
import json
import os
import shutil
import time
from pathlib import Path

import cloudpickle
from IPython.display import display
from PIL import Image

import mobilenet_helpers

cloudpickle.register_pickle_by_value(mobilenet_helpers)

VALID_EXECUTION_MODES = {
    "in-process",
    "python-task",
    "stateful-serverless",
}

environment_mode = os.environ.get("MOBILENET_EXECUTION_MODE")
if environment_mode is None:
    execution_mode = EXECUTION_MODE
    execution_mode_source = "notebook configuration cell"
else:
    execution_mode = environment_mode.strip()
    execution_mode_source = "environment variable MOBILENET_EXECUTION_MODE"

if execution_mode not in VALID_EXECUTION_MODES:
    raise ValueError(
        f"Execution mode must be one of {sorted(VALID_EXECUTION_MODES)}; "
        f"received {execution_mode!r}"
    )

# Floability derives this value from its --no-worker CLI option.
workers_enabled = os.environ.get("FLOABILITY_WORKERS_ENABLED")

if workers_enabled not in {None, "0", "1"}:
    raise ValueError("FLOABILITY_WORKERS_ENABLED must be either '0' or '1'")
if execution_mode == "in-process" and workers_enabled == "1":
    raise RuntimeError(
        "In-process inference requires Floability's --no-worker option"
    )
if execution_mode != "in-process" and workers_enabled == "0":
    raise RuntimeError(
        f"Execution mode {execution_mode!r} requires Floability workers"
    )

MODEL_PATH = Path("data/mobilenetv2-10.onnx")
LABELS_PATH = Path("data/imagenet-synset.txt")
IMAGE_DIR = Path("data/images")
MANIFEST_PATH = Path("data/image-manifest.json")
OUTPUT_DIR = Path("outputs")

print(f"Selected execution mode: {execution_mode}")
print(f"Mode source: {execution_mode_source}")


## Step 3 — Validate the dataset contract and build deterministic batches

The workflow only depends on an image directory and its manifest. The manifest records the dataset identity, version, image count, filenames, and SHA-256 checksums. It is independent of whether Floability staged those files from the backpack or from a remote source.

Images are sorted by filename and divided according to `BATCH_SIZE`. The final batch may contain fewer images, so the workflow supports any positive image count.


In [ ]:
for required_path in (MODEL_PATH, LABELS_PATH):
    if not required_path.is_file():
        raise FileNotFoundError(f"Required staged input not found: {required_path}")

image_paths, manifest = mobilenet_helpers.load_and_verify_images(
    IMAGE_DIR,
    MANIFEST_PATH,
)
image_batches = mobilenet_helpers.make_image_batches(image_paths, BATCH_SIZE)
expected_image_names = [path.name for path in image_paths]

print(f"Dataset: {manifest['dataset_id']}@{manifest['dataset_version']}")
print(f"Verified images: {len(image_paths)}")
print(
    f"Microbatches: {len(image_batches)} with at most "
    f"{BATCH_SIZE} images each"
)


## Step 4 — Identify the inference functions

All modes use the same resize, center-crop, ImageNet normalization, ONNX inference, and top-five postprocessing. `classify_batch_cold` creates a new ONNX Runtime session and is used by both in-process and PythonTask execution. `load_mobilenet_library` initializes persistent state, while `classify_batch_stateful` reuses that state for FunctionCall tasks.


In [ ]:
# Ordinary TaskVine PythonTask: every task creates its own model session.
def classify_batch_cold(model_path, labels_path, batch_dir, top_k):
    import os
    import socket
    import time
    import uuid

    started_at = time.perf_counter()
    session = mobilenet_helpers.create_inference_session(model_path)
    labels = mobilenet_helpers.load_imagenet_labels(labels_path)
    loaded_at = time.perf_counter()
    predictions = mobilenet_helpers.classify_image_directory(
        session, labels, batch_dir, top_k
    )
    return {
        "predictions": predictions,
        "session_load_id": uuid.uuid4().hex[:8],
        "session_load_seconds": loaded_at - started_at,
        "task_seconds": time.perf_counter() - started_at,
        "hostname": socket.gethostname(),
        "pid": os.getpid(),
    }


# TaskVine LibraryTask context: initialize persistent worker state once.
def load_mobilenet_library(model_path, labels_path):
    import os
    import socket
    import time
    import uuid

    started_at = time.perf_counter()
    return {
        "inference_session": mobilenet_helpers.create_inference_session(
            model_path
        ),
        "imagenet_labels": mobilenet_helpers.load_imagenet_labels(
            labels_path
        ),
        "library_load_id": uuid.uuid4().hex[:8],
        "library_load_seconds": time.perf_counter() - started_at,
        "library_hostname": socket.gethostname(),
        "library_pid": os.getpid(),
    }


# TaskVine FunctionCall: retrieve and reuse the LibraryTask state.
def classify_batch_stateful(batch_dir, top_k):
    import os
    import time

    from ndcctools.taskvine.utils import load_variable_from_library

    started_at = time.perf_counter()
    session = load_variable_from_library("inference_session")
    labels = load_variable_from_library("imagenet_labels")
    predictions = mobilenet_helpers.classify_image_directory(
        session, labels, batch_dir, top_k
    )
    return {
        "predictions": predictions,
        "library_load_id": load_variable_from_library("library_load_id"),
        "library_load_seconds": load_variable_from_library(
            "library_load_seconds"
        ),
        "library_hostname": load_variable_from_library("library_hostname"),
        "library_pid": load_variable_from_library("library_pid"),
        "function_pid": os.getpid(),
        "call_seconds": time.perf_counter() - started_at,
    }


## Step 5 — Start the selected execution path

In-process mode deliberately skips TaskVine entirely. A distributed mode creates the manager, materializes the deterministic image batches, and declares the staged inputs that workers need.


In [ ]:
batch_root = None
manager = None
library_reuse = {}

if execution_mode == "in-process":
    print("In-process mode: no TaskVine manager or workers are used")
else:
    import ndcctools.taskvine as vine

    manager_name = os.environ.get("VINE_MANAGER_NAME")
    if not manager_name:
        raise RuntimeError(
            "VINE_MANAGER_NAME is not set; start this distributed mode "
            "through Floability"
        )

    # Floability normalizes --manager-port-range to this comma-separated
    # environment variable before it starts the notebook.
    port_spec = os.environ.get("VINE_MANAGER_PORTS", "9123,9150")
    ports = [
        int(value.strip())
        for value in port_spec.split(",")
        if value.strip()
    ]
    if not ports:
        raise ValueError("VINE_MANAGER_PORTS does not contain a port")
    manager_port = (
        ports[0] if len(ports) == 1 else [min(ports), max(ports)]
    )

    manager = vine.Manager(port=manager_port, name=manager_name)
    manager.tune("watch-library-logfiles", 1)
    batch_root, batch_paths = mobilenet_helpers.materialize_batch_directories(
        image_batches,
        prefix="mobilenet-notebook-batches-",
    )
    declared_model = manager.declare_file(str(MODEL_PATH), cache=True)
    declared_labels = manager.declare_file(str(LABELS_PATH), cache=True)
    declared_batches = {
        batch_path: manager.declare_file(str(batch_path), cache=True)
        for batch_path in batch_paths
    }

    print(f"Manager name: {manager_name}")
    print(f"Manager port: {manager.port}")
    print("Declared the model, labels, and image microbatches")


## Step 6 — Run locally or submit distributed tasks

In-process mode classifies the staged directory directly. PythonTask mode submits one isolated task per batch. Stateful serverless mode first installs a persistent library, then submits FunctionCall tasks that reuse its loaded model.


In [ ]:
if execution_mode == "in-process":
    started_at = time.perf_counter()
    local_result = classify_batch_cold(
        str(MODEL_PATH),
        str(LABELS_PATH),
        str(IMAGE_DIR),
        TOP_K,
    )
    local_result["batch"] = "all-images"
    local_result["worker_address"] = None
    results = [local_result]
    elapsed_seconds = time.perf_counter() - started_at
    task_batches = {}
    print(f"Classified {len(image_paths)} images without workers")
else:
    if execution_mode == "stateful-serverless":
        library = manager.create_library_from_functions(
            LIBRARY_NAME,
            classify_batch_stateful,
            add_env=False,
            exec_mode="direct",
            library_context_info=[
                load_mobilenet_library,
                ["model.onnx", "labels.txt"],
                {},
            ],
        )
        library.add_input(declared_model, "model.onnx")
        library.add_input(declared_labels, "labels.txt")
        library.set_cores(1)
        library.set_function_slots(1)
        manager.install_library(library)
        print(f"Installed persistent library: {LIBRARY_NAME}")

    task_batches = {}
    started_at = time.perf_counter()
    for batch_path in batch_paths:
        if execution_mode == "python-task":
            task = vine.PythonTask(
                classify_batch_cold,
                "model.onnx",
                "labels.txt",
                "batch",
                TOP_K,
            )
            task.add_input(declared_model, "model.onnx")
            task.add_input(declared_labels, "labels.txt")
        else:
            task = vine.FunctionCall(
                LIBRARY_NAME,
                "classify_batch_stateful",
                "batch",
                TOP_K,
            )

        task.add_input(declared_batches[batch_path], "batch")
        task.set_cores(1)
        task_id = manager.submit(task)
        task_batches[task_id] = batch_path.name

    print(f"Submitted {len(task_batches)} tasks using {execution_mode}")


## Step 7 — Collect distributed results

The in-process result is already available. For a distributed run, the manager records each worker address and fails the workflow if any batch fails. Stateful calls also verify that inference ran inside the persistent library process.


In [ ]:
if execution_mode != "in-process":
    results = []
    failures = []
    while not manager.empty():
        completed = manager.wait(5)
        if not completed:
            continue
        if not completed.successful():
            failures.append((completed.id, completed.result))
            print(f"FAILED task={completed.id} result={completed.result}")
            continue

        result = completed.output
        if execution_mode == "stateful-serverless":
            if result["function_pid"] != result["library_pid"]:
                raise RuntimeError(
                    "FunctionCall did not execute inside its library process"
                )
            execution_detail = f"load_id={result['library_load_id']}"
        else:
            execution_detail = f"session={result['session_load_id']}"

        result["task_id"] = completed.id
        result["batch"] = task_batches[completed.id]
        result["worker_address"] = completed.addrport
        results.append(result)
        print(
            f"task={completed.id} batch={result['batch']} "
            f"{execution_detail} worker={completed.addrport}"
        )

    if failures:
        raise RuntimeError(f"Inference task failures: {failures}")
    if len(results) != len(batch_paths):
        raise RuntimeError(
            f"Expected {len(batch_paths)} results; received {len(results)}"
        )
    elapsed_seconds = time.perf_counter() - started_at

print(f"Execution completed in {elapsed_seconds:.2f} seconds")


## Step 8 — Validate and inspect predictions

Every manifest image must appear exactly once in the results. For stateful execution, repeated load IDs show that multiple FunctionCalls reused the same initialized library instance. Whether reuse occurs in this small run also depends on worker count and scheduling.


In [ ]:
predictions = mobilenet_helpers.predictions_by_image(
    results,
    expected_image_names,
)

if execution_mode == "stateful-serverless":
    library_reuse = mobilenet_helpers.library_reuse_by_id(results)
    print(f"Persistent library instances: {len(library_reuse)}")
    for load_id, batches in sorted(library_reuse.items()):
        print(f"  {load_id}: {len(batches)} microbatch(es)")
elif execution_mode == "python-task":
    print(f"Independent ONNX sessions: {len(results)}")
else:
    print("One ONNX session ran in the notebook process")

print()
print(f"{'Image':<27} {'Top prediction':<40} Confidence")
print("-" * 82)
for image_name in sorted(predictions):
    top_prediction = predictions[image_name]["top_predictions"][0]
    print(
        f"{image_name:<27} "
        f"{top_prediction['label'][:39]:<40} "
        f"{top_prediction['probability']:.1%}"
    )


## Step 9 — Save and visualize the run

The summary records the dataset version, selected mode, timing, predictions, and available execution metadata. The contact sheet displays at most the first 24 deterministically ordered images so it remains usable for larger future profiles.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
summary = {
    "dataset_id": manifest["dataset_id"],
    "dataset_version": manifest["dataset_version"],
    "execution_mode": execution_mode,
    "execution_mode_source": execution_mode_source,
    "image_count": len(image_paths),
    "batch_size": BATCH_SIZE,
    "batch_count": len(image_batches),
    "elapsed_seconds": elapsed_seconds,
    "distinct_library_loads": (
        len(library_reuse)
        if execution_mode == "stateful-serverless"
        else None
    ),
    "task_results": results,
}
summary_path = OUTPUT_DIR / f"{execution_mode}-summary.json"
summary_path.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")

contact_sheet_path = OUTPUT_DIR / f"{execution_mode}-contact-sheet.jpg"
displayed_count = mobilenet_helpers.save_contact_sheet(
    image_paths,
    predictions,
    contact_sheet_path,
)

if batch_root is not None:
    shutil.rmtree(batch_root)

print("=" * 72)
print("MOBILENET BATCH INFERENCE COMPLETE")
print(f"Execution mode: {execution_mode}")
print(f"Validated images: {len(predictions)}")
print(f"Elapsed time: {elapsed_seconds:.2f} seconds")
print(f"Results: {summary_path}")
print(f"Contact sheet: {contact_sheet_path} ({displayed_count} images shown)")
print("=" * 72)

display(Image.open(contact_sheet_path))


## Interpreting the three modes

- **In-process:** provides a worker-free installation and data smoke test.
- **PythonTask:** provides distributed task isolation, but every microbatch pays the model initialization cost.
- **Stateful Serverless Computing:** each library instance pays initialization once and can reuse its ONNX session across later FunctionCalls.

This run is a functional demonstration rather than a formal performance benchmark. Distributed timings include scheduling and file transfers, and the in-process mode does not include those costs.
